In [19]:
import numpy as np
import scipy.io.wavfile as wavfile
import torch
from scipy.io import wavfile
from training.training_process import create_training_config

config = create_training_config()

model = config.model.generator

point = 11

model.load_state_dict(torch.load(f"./checkpoints/checkpoint-{point}00.pth", map_location=config.device))
model.eval()  # CRITICAL: Sets dropout/batchnorm to inference mode


def generate_speech(text, output_path="output.wav"):
    print(f"Generating speech for: '{text}'")

    # 1. Tokenize
    encoding = config.tokenizer(text, return_tensors="pt")
    input_ids = encoding["input_ids"].to(config.device)

    # 2. Generate (training=False triggers duration prediction)
    with torch.no_grad():
        output = model(input_ids, training=False)
    audio_tensor = output['audio_generated']
    durs = output['durs']
    print(f"Final durations: {durs.squeeze().cpu().numpy()}")
    print(f"Total mel frames: {durs.sum().item()}")

    # 3. Post-process audio
    # audio_tensor is [1, 1, Time]. Squeeze to [Time]
    audio_np = audio_tensor.squeeze().cpu().numpy()

    # Ensure it's in [-1, 1] range, then scale to 16-bit PCM
    audio_np = np.clip(audio_np, -1.0, 1.0)
    audio_int16 = (audio_np * 32767).astype(np.int16)

    # 4. Save to WAV
    sample_rate = 16000
    wavfile.write(output_path, sample_rate, audio_int16)
    print(f"✅ Successfully saved to: {output_path}")


test_text = "Привет! Это тест синтезированной речи на чистой архитектуре пайторч."
generate_speech(test_text, f"my_first_tts_output_{point}.wav")

Generating speech for: 'Привет! Это тест синтезированной речи на чистой архитектуре пайторч.'
Final durations: [ 4  3  7  3 16  9  6  3 11  5  7  5  4  3 11  3 11  3 10  5  3  2 18  3
  6  6  6  3 10  5  3  3 14  3  7  7  5  3 10  3  6  5  5  3  9  9  4  3
 23  4  7  3 16  8  5  3  8  3  9  3  3  5  3  3  9  3  9  7  3  3  8 11
  2  3 14  3  8 14  2  3  6  3  9  6  6  3 11  3 11  3  4  5  3  3 16  7
  4  3 13  3 10  6  7  3  7  7  4  3 14  4  6  3  6  3 10 10  2  3  8  3
 10  7  4  3 11  3 12  3  3  3  3  2]
Total mel frames: 806
✅ Successfully saved to: my_first_tts_output_11.wav
